In [148]:
import time
import openpyxl
import pandas as pd
import numpy as np
import seaborn as sns
import requests
import matplotlib.pyplot as plt

In [149]:
c_bonds_data = pd.read_excel('/Users/vladislav/Desktop/Данные/CBonds.xlsx',engine='openpyxl',sheet_name='Data')

In [150]:
c_bonds_data.head()

,TRADEDATE,Urals,Brent,GDB(4 months),InterestRate,RUONIA,USD/RUB,EUR/RUB,STB,RGBI,RVI,EUR/USD,BankLiqudity,RTS,PMI,RUABITR,VIX,DXY
0,1997-11-22,No data,No data,No data,No data,No data,No data,No data,No data,No data,No data,No data,No data,No data,No data,No data,No data,No data
1,1997-11-23,No data,No data,No data,No data,No data,No data,No data,No data,No data,No data,No data,No data,No data,No data,No data,No data,No data
2,1997-11-24,No data,"19,09",No data,No data,No data,No data,No data,No data,No data,No data,No data,No data,"368,47",No data,No data,"29,8","96,68"
3,1997-11-25,No data,"19,09",No data,No data,No data,No data,No data,No data,No data,No data,No data,No data,"347,04",No data,No data,"28,95","97,23"
4,1997-11-26,No data,"18,56",No data,No data,No data,No data,No data,No data,No data,No data,No data,No data,"356,81",No data,No data,"27,99","97,64"


# Парсим MOEX

In [151]:
imoex_code = "IMOEX"
start_date = "1997-09-22"   #начало истории IMOEX

end_date = "2026-04-14"     #  сегодняшняя дата
batch_size = 100             # постраничная загрузка (по 100 записей за раз)

# Список для хранения данных
all_data = []

# Начало пагинации
start = 0

while True:
    url = f"https://iss.moex.com/iss/history/engines/stock/markets/index/securities/{imoex_code}.json"
    params = {
        "from": start_date,
        "till": end_date,
        "start": start
    }

    response = requests.get(url, params=params)
    if response.status_code != 200:
        print(f"Ошибка запроса: {response.status_code}")
        break

    data_json = response.json()
    # Получаем колонки и строки данных
    columns = data_json['history']['columns']
    rows = data_json['history']['data']

    if not rows:
        break  # если данных больше нет, выходим

    # Добавляем в общий список
    all_data.extend(rows)

    # Работа с пагинацией
    cursor = data_json.get('history.cursor', {}).get('data', [[0, 0]])
    if cursor[0][0] >= cursor[0][1]:
        break  # достигли конца

    start = cursor[0][0] + batch_size

# Преобразуем в DataFrame
IMOEX_DATA_with_target = pd.DataFrame(all_data, columns=columns)

# Оставляем только важные поля
IMOEX_DATA_with_target = IMOEX_DATA_with_target[["TRADEDATE", "OPEN", "HIGH", "LOW", "CLOSE"]]

# Преобразуем дату
IMOEX_DATA_with_target["TRADEDATE"] = pd.to_datetime(IMOEX_DATA_with_target["TRADEDATE"])
IMOEX_DATA_with_target = IMOEX_DATA_with_target.sort_values("TRADEDATE")

# Парсим RGBI

In [152]:

start_date = "1997-09-22"
end_date = "2026-04-14"
url = (
    "https://iss.moex.com/iss/history/engines/stock/"
    "markets/index/securities/RGBI.json"
)


all_data = []

# ==========================================================
# PAGINATION
# ==========================================================
start = 0
page_size = 100

while True:

    params = {
        'from': start_date,
        'till': end_date,

        'iss.meta': 'off',
        'iss.only': 'history',

        'history.columns': 'TRADEDATE,CLOSE',

        'start': start
    }

    response = requests.get(url, params=params)

    data = response.json()

    history_data = data['history']['data']

    # если пусто -> конец
    if not history_data:
        break

    all_data.extend(history_data)

    start += page_size
    time.sleep(0.3)

# ==========================================================
# DATAFRAME
# ==========================================================
df_RGBI = pd.DataFrame(
    all_data,
    columns=['TRADEDATE', 'RGBI']
)

# ==========================================================
# DATETIME
# ==========================================================
df_RGBI['TRADEDATE'] = pd.to_datetime(
    df_RGBI['TRADEDATE']
)

# ==========================================================
# SORT
# ==========================================================
df_RGBI = df_RGBI.sort_values(
    'TRADEDATE'
)


In [153]:
df_RGBI.head()

,TRADEDATE,RGBI
0,2002-12-30,100.00
1,2003-01-04,100.16
2,2003-01-05,100.88
3,2003-01-08,101.03
4,2003-01-09,101.05


$$
\text{crises rate}_{i,t} = \frac{P_t}{\max(P_{t-T}, \dots, P_t)} \text{, где T = 19}
$$

$$
\text{crises}_{i,t} =
\begin{cases}
1, & \text{если } \text{crises rate}_{i,t} < \overline{\text{crises rate}_i} - \sigma_{i,t} \\
0, & \text{иначе}
\end{cases}
$$


Где
$$
\sigma_{i,t} \quad \text{— стандартное отклонение crises rate}_{i,t} \text{ за последние 20 дней}, \\
\overline{\text{crises rate}_i} \quad \text{— средний crises rate}_{i,t} \text{ за последние 20 дней.}
$$

In [154]:
IMOEX_DATA_with_target_copy = IMOEX_DATA_with_target.copy()
crises_rate = []
'''
Расчитываем индекс кризиса по формуле выше
'''
for j,i in enumerate(range(19,IMOEX_DATA_with_target_copy.shape[0]),start=19):
    cr = []
    cr = [IMOEX_DATA_with_target.iloc[k,-1] for k in range(j-19,j+1)]
    crises_rate.append(IMOEX_DATA_with_target.iloc[j,-1]/max(cr))
crises_rate = np.array(crises_rate)
mean_cr_rate = [crises_rate[0+i:20+i].mean() for i in range(20,len(crises_rate))]
std_cr_rate = [crises_rate[0+i:20+i].std() for i in range(20,len(crises_rate))]
crisis = crises_rate[20:len(crises_rate)] < np.array(mean_cr_rate) - np.array(std_cr_rate)
crisis = crisis.astype('int')
crisis = list(crisis)
l = [np.nan]*40 # плюс один чтобы кризис был смещен на один день
l.extend(crisis)
l.pop(-1)
crisis = np.array(l)
IMOEX_DATA_with_target['crisis'] = crisis

In [155]:
IMOEX_DATA_with_target['crisis'].mean()

np.float64(0.24401913875598086)

In [156]:
IMOEX_DATA_with_target.set_index('TRADEDATE',inplace=True)


In [157]:
IMOEX_DATA_with_target.tail()
IMOEX_DATA_with_target = IMOEX_DATA_with_target.loc['1997-11-22':,:]
IMOEX_DATA_with_target.reset_index(inplace=True)

In [158]:
IMOEX_DATA_with_target.head()

,TRADEDATE,OPEN,HIGH,LOW,CLOSE,crisis
0,1997-11-24,NaN,NaN,NaN,74.01,1.0
1,1997-11-25,NaN,NaN,NaN,69.49,1.0
2,1997-11-26,NaN,NaN,NaN,72.65,1.0
3,1997-11-27,NaN,NaN,NaN,70.20,1.0
4,1997-11-28,NaN,NaN,NaN,67.94,1.0


# Делаем merge

In [159]:
IMOEX_DATA_with_target.head()

,TRADEDATE,OPEN,HIGH,LOW,CLOSE,crisis
0,1997-11-24,NaN,NaN,NaN,74.01,1.0
1,1997-11-25,NaN,NaN,NaN,69.49,1.0
2,1997-11-26,NaN,NaN,NaN,72.65,1.0
3,1997-11-27,NaN,NaN,NaN,70.20,1.0
4,1997-11-28,NaN,NaN,NaN,67.94,1.0


In [160]:
c_bonds_data.head()

,TRADEDATE,Urals,Brent,GDB(4 months),InterestRate,RUONIA,USD/RUB,EUR/RUB,STB,RGBI,RVI,EUR/USD,BankLiqudity,RTS,PMI,RUABITR,VIX,DXY
0,1997-11-22,No data,No data,No data,No data,No data,No data,No data,No data,No data,No data,No data,No data,No data,No data,No data,No data,No data
1,1997-11-23,No data,No data,No data,No data,No data,No data,No data,No data,No data,No data,No data,No data,No data,No data,No data,No data,No data
2,1997-11-24,No data,"19,09",No data,No data,No data,No data,No data,No data,No data,No data,No data,No data,"368,47",No data,No data,"29,8","96,68"
3,1997-11-25,No data,"19,09",No data,No data,No data,No data,No data,No data,No data,No data,No data,No data,"347,04",No data,No data,"28,95","97,23"
4,1997-11-26,No data,"18,56",No data,No data,No data,No data,No data,No data,No data,No data,No data,No data,"356,81",No data,No data,"27,99","97,64"


In [161]:
df = c_bonds_data.merge(IMOEX_DATA_with_target, on='TRADEDATE', how='outer')

In [162]:
df.isna().sum()

TRADEDATE           0
Urals               0
Brent               0
GDB(4 months)       0
InterestRate        0
RUONIA              0
USD/RUB             0
EUR/RUB             0
STB                 0
RGBI                0
RVI                 0
EUR/USD             0
BankLiqudity        0
RTS                 0
PMI                 0
RUABITR             0
VIX                 0
DXY                 0
OPEN             4530
HIGH             4521
LOW              4521
CLOSE            3269
crisis           3269
dtype: int64

In [163]:
df.replace("No data", np.nan,inplace=True)

In [164]:
df.head()

,TRADEDATE,Urals,Brent,GDB(4 months),InterestRate,RUONIA,USD/RUB,EUR/RUB,STB,RGBI,...,RTS,PMI,RUABITR,VIX,DXY,OPEN,HIGH,LOW,CLOSE,crisis
0,1997-11-22,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,1997-11-23,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,1997-11-24,NaN,"19,09",NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,"368,47",NaN,NaN,"29,8","96,68",NaN,NaN,NaN,74.01,1.0
3,1997-11-25,NaN,"19,09",NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,"347,04",NaN,NaN,"28,95","97,23",NaN,NaN,NaN,69.49,1.0
4,1997-11-26,NaN,"18,56",NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,"356,81",NaN,NaN,"27,99","97,64",NaN,NaN,NaN,72.65,1.0


In [165]:
#df.drop(['InterestRate','GDB(4 months)','RUONIA','RVI','RUABITR','BankLiqudity','STB','RGBI','PMI'],axis=1,#inplace=True)
df.drop(['InterestRate','GDB(4 months)','RVI','RUABITR','BankLiqudity','STB','RGBI','PMI'],axis=1,inplace=True)
df = df.merge(df_RGBI, on='TRADEDATE', how='outer')

In [166]:
df.set_index('TRADEDATE',inplace=True)


In [167]:
df.shape

(10371, 15)

In [168]:
df = df.loc['2010-01-11':,:] #2010-01-01 начало расчёта RUONIA, 2010-01-11 первые доступные данные по RUONIA
#df = df.loc['1999-03-31':,:]

In [169]:
df.head()

,Urals,Brent,RUONIA,USD/RUB,EUR/RUB,EUR/USD,RTS,VIX,DXY,OPEN,HIGH,LOW,CLOSE,crisis,RGBI
TRADEDATE,,,,,,,,,,,,,,,
2010-01-11,NaN,"80,97","2,98",NaN,NaN,"1,4528","1553,06","17,55","77,005",1370.30,1456.78,1370.30,1444.78,0.0,130.32
2010-01-12,NaN,"79,3","3,1","29,4283","42,6681","1,4481","1535,78","18,25","76,954",1444.78,1445.70,1424.73,1427.67,0.0,129.02
2010-01-13,NaN,"78,31","3,4","29,3774","42,6149","1,4563","1538,43","17,85","76,846",1427.62,1444.86,1412.87,1435.01,0.0,129.68
2010-01-14,NaN,"77,82","3,59","29,6409","42,9497","1,4486","1561,91","17,63","76,732",1435.01,1456.34,1435.01,1455.65,0.0,128.88
2010-01-15,NaN,"77,11","4,21","29,4299","42,7764","1,4374","1559,25","17,91","77,323",1455.47,1469.55,1448.68,1452.67,0.0,129.87


# Заполняем пропуски

In [170]:
for i in range(0,df.shape[0]):
    if df.iloc[i,0] is np.nan:
        df.iloc[i,0] = df.iloc[i,1]

In [171]:
df.head()

,Urals,Brent,RUONIA,USD/RUB,EUR/RUB,EUR/USD,RTS,VIX,DXY,OPEN,HIGH,LOW,CLOSE,crisis,RGBI
TRADEDATE,,,,,,,,,,,,,,,
2010-01-11,"80,97","80,97","2,98",NaN,NaN,"1,4528","1553,06","17,55","77,005",1370.30,1456.78,1370.30,1444.78,0.0,130.32
2010-01-12,"79,3","79,3","3,1","29,4283","42,6681","1,4481","1535,78","18,25","76,954",1444.78,1445.70,1424.73,1427.67,0.0,129.02
2010-01-13,"78,31","78,31","3,4","29,3774","42,6149","1,4563","1538,43","17,85","76,846",1427.62,1444.86,1412.87,1435.01,0.0,129.68
2010-01-14,"77,82","77,82","3,59","29,6409","42,9497","1,4486","1561,91","17,63","76,732",1435.01,1456.34,1435.01,1455.65,0.0,128.88
2010-01-15,"77,11","77,11","4,21","29,4299","42,7764","1,4374","1559,25","17,91","77,323",1455.47,1469.55,1448.68,1452.67,0.0,129.87


In [172]:
df.shape

(5938, 15)

In [173]:
df = df[df['crisis'].notna()]

In [174]:
df.to_excel('data_all.xlsx')

In [175]:
df.shape

(4085, 15)

In [176]:
df = pd.read_excel('/Users/vladislav/Desktop/Данные/data_all.xlsx',decimal=',',index_col='TRADEDATE')

In [177]:
df.head()

,Urals,Brent,RUONIA,USD/RUB,EUR/RUB,EUR/USD,RTS,VIX,DXY,OPEN,HIGH,LOW,CLOSE,crisis,RGBI
TRADEDATE,,,,,,,,,,,,,,,
2010-01-11,80.97,80.97,2.98,NaN,NaN,1.4528,1553.06,17.55,77.005,1370.30,1456.78,1370.30,1444.78,0,130.32
2010-01-12,79.30,79.30,3.10,29.4283,42.6681,1.4481,1535.78,18.25,76.954,1444.78,1445.70,1424.73,1427.67,0,129.02
2010-01-13,78.31,78.31,3.40,29.3774,42.6149,1.4563,1538.43,17.85,76.846,1427.62,1444.86,1412.87,1435.01,0,129.68
2010-01-14,77.82,77.82,3.59,29.6409,42.9497,1.4486,1561.91,17.63,76.732,1435.01,1456.34,1435.01,1455.65,0,128.88
2010-01-15,77.11,77.11,4.21,29.4299,42.7764,1.4374,1559.25,17.91,77.323,1455.47,1469.55,1448.68,1452.67,0,129.87


# Сохраняем данные на предсказания на 1-7 дней вперед

In [178]:
for i in range(1,8):
    df = pd.read_excel('/Users/vladislav/Desktop/Данные/data_all.xlsx',decimal=',',index_col='TRADEDATE')
    df['crisis'] = df['crisis'].shift(i)
    df = df[df['crisis'].notna()]
    df.to_excel(f'predictions_for_{i}_days.xlsx')